# 08 · Red neuronal (TensorFlow/Keras) y comparación de modelos

**Pregunta:** ¿una red neuronal recurrente pronostica mejor que el modelo de árboles del notebook 06?

**Comparación justa:** mismo problema (cambio a 24 h de cloro, turbidez y pH; la turbidez en escala
logarítmica), mismas particiones, misma meta (`calidad_agua/metas.py`) y la misma información disponible. Lo
que cambia es cómo la recibe cada modelo:

| | Árboles (scikit-learn) | Red recurrente (Keras) |
|---|---|---|
| Entrada | ~84 resúmenes calculados a mano (medias, cambios, valores de hace *k* horas) | la **secuencia** de las últimas 48 h (23 variables por hora) |
| Modelo | `HistGradientBoostingRegressor`, uno por parámetro | **LSTM** de 48 unidades + capa densa, un solo modelo para los tres parámetros |
| Sensor | variable categórica | *embedding* de 4 dimensiones |

**Un ajuste que salió de las pruebas:** entrenar **solo con horas normales** (sin evento en la hora
pronosticada). Los cambios bruscos de los eventos no se pueden anticipar a 24 h; si el modelo los ve
al entrenar, intenta memorizarlos y empeora en el día a día. Los eventos los atiende el detector
(notebook 07). Para que la comparación sea justa, los árboles también se reentrenan así.

**Entradas y objetivos se limpian distinto, a propósito.** Las **entradas** se limpian solo con las
reglas que existen en operación (presión, rango físico y sensor pegado:
`calidad.lecturas_operativas`), porque en vivo no hay etiquetas de falla y el modelo tiene que
aprender a convivir con lecturas con deriva, picos o ruido. Los **objetivos** son el valor real,
limpio de fallas del sensor (`calidad.lecturas_validas`), porque lo que se quiere pronosticar es el
agua, no la falla del instrumento.

**Cómo se decide:** se comparan en **validación**, se elige el mejor modelo **por parámetro** y
recién después se evalúa una vez en **prueba**. Se incluye una tercera opción: la **combinación**
(promedio de los dos pronósticos), que suele ser más estable que cada modelo por separado.

In [1]:
import os, sys, time
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # oculta los mensajes internos de TensorFlow
sys.path.insert(0, str(Path.cwd().parent))          # por si calidad_agua no está instalado (pip install -e .)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras

from calidad_agua import carga, calidad, metas, modelo, variables, red_neuronal as rn
from calidad_agua.graficos import estilo, guardar, COLORES, GRIS

estilo()
pd.set_option("display.precision", 1)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

crudo = carga.cargar_todo()
t15 = calidad.lecturas_operativas(crudo)        # ENTRADAS: limpieza que también existe en operación
t15_real = calidad.lecturas_validas(crudo)      # OBJETIVOS: valor real, sin fallas del sensor
h = variables.horario(t15)
h_real = variables.horario(t15_real)
tabla, tipicos = variables.tabla(h, horizonte=metas.HORIZONTE_H, h_objetivo=h_real)
X = variables.columnas_entrada(tabla)
datos = rn.preparar(h, tipicos, h_objetivo=h_real)
PARAM = rn.SALIDAS
ETIQ = {"CL2": "Cloro libre", "TURB": "Turbidez", "PH": "pH"}
print(f"Secuencias: {len(datos['sensores'])} sensores × {len(datos['fechas']):,} horas × "
      f"{datos['secuencias'][0].shape[1]} variables por hora · ventana de {rn.VENTANA_H} h")
print("Variables por hora:", ", ".join(datos["variables"]))

Secuencias: 21 sensores × 17,520 horas × 23 variables por hora · ventana de 48 h
Variables por hora: CL2, TURB, PH, TEMP, P, hora_sin, hora_cos, anio_sin, anio_cos, lluvia, caudal, fuente_CL2, red_CL2, fuente_TURB, red_TURB, fuente_PH, red_PH, fuente_TEMP, red_TEMP, falta_CL2, falta_TURB, falta_PH, falta_TEMP


## 1. Red neuronal LSTM

Se entrena tres veces con distinta semilla aleatoria y se promedian los pronósticos. Una red
neuronal da resultados algo distintos en cada entrenamiento; promediar tres reduce esa variación y
muestra cuánto cambia de una vez a otra.

El entrenamiento se detiene cuando el error en una muestra de validación deja de bajar durante 2
épocas, y se conservan los pesos de la mejor época.

In [2]:
idx_entr = rn.indices(datos, "entrenamiento", solo_normal=True)
idx_val_parada = rn.indices(datos, "validacion", solo_normal=True)[::3]
print(f"Ejemplos de entrenamiento: {len(idx_entr):,}")

redes, historias = [], []
for semilla in [0, 1, 2]:
    keras.utils.set_random_seed(semilla)
    red = rn.construir(datos["secuencias"][0].shape[1], len(datos["sensores"]))
    red.optimizer.learning_rate.assign(3e-4)
    t0 = time.time()
    hist = red.fit(rn.Lotes(datos, idx_entr, 512, barajar=True, semilla=semilla),
                   validation_data=rn.Lotes(datos, idx_val_parada, 2048), epochs=8, verbose=0,
                   callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
    redes.append(red); historias.append(hist.history)
    print(f"semilla {semilla}: {len(hist.history['loss'])} épocas, mejor la {int(np.argmin(hist.history['val_loss'])) + 1} "
          f"· {time.time() - t0:.0f} s")
redes[0].summary()

Ejemplos de entrenamiento: 226,801


semilla 0: 6 épocas, mejor la 4 · 125 s


semilla 1: 4 épocas, mejor la 2 · 84 s


semilla 2: 5 épocas, mejor la 3 · 103 s


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ secuencia           │ (None, 48, 23)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_sensor    │ (None, 1, 4)      │         84 │ sensor[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 48)        │     13,824 │ secuencia[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4)         │          0 │ embedding_sensor… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tipicos             │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 55)        │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ flatten[0][0],    │
│                     │                   │            │ tipicos[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 55)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      3,584 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cambio_24h (Dense)  │ (None, 3)         │        195 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 53,063 (207.28 KB)

 Trainable params: 17,687 (69.09 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 35,376 (138.19 KB)

In [3]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for k, hst in enumerate(historias):
    ep = np.arange(1, len(hst["loss"]) + 1)
    ax.plot(ep, hst["loss"], color=COLORES[k], lw=1.2, label=f"entrenamiento (semilla {k})")
    ax.plot(ep, hst["val_loss"], color=COLORES[k], lw=1.2, ls="--", label=f"validación (semilla {k})")
ax.set_xlabel("época"); ax.set_ylabel("error absoluto (escala normalizada)")
ax.set_xticks(range(1, max(len(h_["loss"]) for h_ in historias) + 1))
ax.set_title("Curvas de aprendizaje de la LSTM")
ax.legend(fontsize=6, ncol=2)
guardar(fig, "08_curvas_aprendizaje"); plt.show()

## 2. Árboles, entrenados también solo con horas normales

In [4]:
entr = tabla["particion"] == "entrenamiento"
arboles = {p: modelo.entrenar(tabla, p, X, entr & (tabla["evento_objetivo"] == 0)) for p in PARAM}
print({p: m.n_iter_ for p, m in arboles.items()})

{'CL2': 400, 'TURB': 400, 'PH': 400}


## 3. Pronósticos de todos los modelos en las mismas filas

Para comparar en igualdad de condiciones, cada modelo pronostica exactamente los mismos pares
(sensor, hora).

In [5]:
def pronosticos(particion):
    idx = rn.indices(datos, particion)
    marco = pd.DataFrame({"nodo": np.array(datos["sensores"])[idx[:, 0]], "fecha": datos["fechas"][idx[:, 1]]})
    marco["evento"] = np.nan_to_num(datos["evento"][idx[:, 1] + metas.HORIZONTE_H, idx[:, 0]])
    lstm = np.mean([rn.pronosticar(r, datos, idx) for r in redes], axis=0)
    filas = marco.merge(tabla, on=["fecha", "nodo"], how="left", suffixes=("", "_t"))
    for k, p in enumerate(PARAM):
        marco[f"obs_{p}"] = filas[f"objetivo_{p}"].values
        marco[f"base_{p}"] = filas[p].values
        marco[f"arboles_{p}"] = modelo.pronosticar(arboles[p], filas, p, X)
        marco[f"lstm_{p}"] = lstm[:, k]
        if p == "TURB":   # en turbidez se promedia en escala logarítmica, coherente con la tolerancia relativa
            marco[f"combinado_{p}"] = np.exp((np.log(np.clip(marco[f"arboles_{p}"], 0.05, None)) +
                                              np.log(np.clip(marco[f"lstm_{p}"], 0.05, None))) / 2)
        else:
            marco[f"combinado_{p}"] = (marco[f"arboles_{p}"] + marco[f"lstm_{p}"]) / 2
    return marco

MODELOS = {"base": "línea base (igual que ayer)", "arboles": "árboles (scikit-learn)",
           "lstm": "LSTM (Keras)", "combinado": "combinación (promedio)"}

def tabla_acierto(marco, condicion="normal"):
    m = (marco["evento"] == 0) if condicion == "normal" else (marco["evento"] > 0)
    out = {}
    for clave, nombre in MODELOS.items():
        out[nombre] = {p: 100 * np.nanmean(metas.acierto(p, marco.loc[m, f"obs_{p}"],
                                                         marco.loc[m, f"{clave}_{p}"])[
                           marco.loc[m, f"{clave}_{p}"].notna() & marco.loc[m, f"base_{p}"].notna()])
                       for p in PARAM}
    return pd.DataFrame(out).T

pron_val = pronosticos("validacion")
acierto_val = tabla_acierto(pron_val)
acierto_val

,CL2,TURB,PH
línea base (igual que ayer),80.9,62.5,84.2
árboles (scikit-learn),89.4,65.9,86.6
LSTM (Keras),87.0,71.4,86.6
combinación (promedio),89.3,70.0,87.7


## 4. Elección por parámetro (validación)

Para cada parámetro se elige el modelo con mayor acierto en validación, en operación normal.

In [6]:
candidatos = acierto_val.drop(index=MODELOS["base"])
elegido = {p: candidatos[p].idxmax() for p in PARAM}
clave_elegida = {p: [k for k, v in MODELOS.items() if v == elegido[p]][0] for p in PARAM}
pd.Series(elegido, name="modelo elegido").to_frame()

,modelo elegido
CL2,árboles (scikit-learn)
TURB,LSTM (Keras)
PH,combinación (promedio)


In [7]:
fig, ax = plt.subplots(figsize=(8.5, 3.4))
x = np.arange(len(PARAM)); ancho = 0.2
colores = {MODELOS["base"]: GRIS, MODELOS["arboles"]: COLORES[0], MODELOS["lstm"]: COLORES[1],
           MODELOS["combinado"]: COLORES[2]}
for k, nombre in enumerate(MODELOS.values()):
    v = acierto_val.loc[nombre, PARAM].values
    barras = ax.bar(x + (k - 1.5) * ancho, v, ancho * 0.92, color=colores[nombre], label=nombre)
    for b_, v_ in zip(barras, v):
        ax.text(b_.get_x() + b_.get_width() / 2, v_ + 1, f"{v_:.0f}", ha="center", fontsize=7, color="#52514e")
ax.axhline(metas.META_PCT, color=COLORES[7], ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels([ETIQ[p] for p in PARAM])
ax.set_ylim(0, 105); ax.set_ylabel("% dentro de la tolerancia")
ax.set_title(f"Validación ({carga.texto_periodo('validacion')}), operación normal")
ax.legend(ncol=2, fontsize=7, loc="lower left")
guardar(fig, "08_comparacion_validacion"); plt.show()

## 5. Evaluación final en prueba

Con el modelo elegido para cada parámetro. Se muestran también los demás, como referencia, pero la
elección ya quedó fija en el paso anterior.

In [8]:
pron_prueba = pronosticos("prueba")
acierto_prueba = tabla_acierto(pron_prueba)
acierto_eventos = tabla_acierto(pron_prueba, "evento")
final = pd.DataFrame({
    "modelo elegido": elegido,
    "% acierto (normal)": {p: acierto_prueba.loc[elegido[p], p] for p in PARAM},
    "línea base": {p: acierto_prueba.loc[MODELOS["base"], p] for p in PARAM},
    "% acierto (eventos)": {p: acierto_eventos.loc[elegido[p], p] for p in PARAM},
})
final["cumple la meta"] = (final["% acierto (normal)"] >= metas.META_PCT) & (final["% acierto (normal)"] > final["línea base"])
final

,modelo elegido,% acierto (normal),línea base,% acierto (eventos),cumple la meta
CL2,árboles (scikit-learn),85.9,78.9,49.6,True
TURB,LSTM (Keras),41.7,42.2,18.8,False
PH,combinación (promedio),80.3,77.6,56.2,True


In [9]:
acierto_prueba

,CL2,TURB,PH
línea base (igual que ayer),78.9,42.2,77.6
árboles (scikit-learn),85.9,36.6,78.3
LSTM (Keras),84.2,41.7,77.0
combinación (promedio),86.0,39.6,80.3


In [10]:
fig, ax = plt.subplots(figsize=(8.5, 3.4))
for k, nombre in enumerate(MODELOS.values()):
    v = acierto_prueba.loc[nombre, PARAM].values
    barras = ax.bar(x + (k - 1.5) * ancho, v, ancho * 0.92, color=colores[nombre], label=nombre,
                    edgecolor=["#0b0b0b" if elegido[p] == nombre else "none" for p in PARAM], linewidth=1.5)
    for b_, v_ in zip(barras, v):
        ax.text(b_.get_x() + b_.get_width() / 2, v_ + 1, f"{v_:.0f}", ha="center", fontsize=7, color="#52514e")
ax.axhline(metas.META_PCT, color=COLORES[7], ls="--", lw=1)
ax.text(-0.55, metas.META_PCT + 1, "meta 80 %", color=COLORES[7], fontsize=8, ha="left")
ax.set_xticks(x); ax.set_xticklabels([f"{ETIQ[p]}\n({metas.texto_tolerancia(p)})" for p in PARAM])
ax.set_ylim(0, 105); ax.set_ylabel("% dentro de la tolerancia")
ax.set_title(f"Prueba ({carga.texto_periodo('prueba')}), operación normal · borde negro = modelo elegido")
ax.legend(ncol=2, fontsize=7, loc="lower left")
guardar(fig, "08_resultado_prueba"); plt.show()

## 6. Guardar

In [11]:
carpeta = Path.cwd().parent / "modelos"
carpeta.mkdir(exist_ok=True)
for k, red in enumerate(redes):
    red.save(carpeta / f"lstm_24h_semilla{k}.keras")
import joblib
joblib.dump({"arboles": arboles, "columnas": X, "tipicos": tipicos, "elegido": clave_elegida,
             "normalizacion": datos["normalizacion"], "variables_lstm": datos["variables"],
             "redes": [f"lstm_24h_semilla{k}.keras" for k in range(len(redes))]},
            carpeta / "pronostico_24h_final.joblib", compress=3)
print("Modelo elegido por parámetro:", clave_elegida)

Modelo elegido por parámetro: {'CL2': 'arboles', 'TURB': 'lstm', 'PH': 'combinado'}


## 7. Conclusiones

**Resultado final en prueba (may–ago 2026, época lluviosa), operación normal**

| Parámetro | Modelo elegido (en validación) | Acierto | Línea base | Notebook 06 | ¿Meta? |
|---|---|---|---|---|---|
| Cloro libre | árboles (scikit-learn) | **86 %** | 79 % | 86 % | **Sí** |
| Turbidez | LSTM (Keras) | 42 % | 42 % | 37 % | No |
| pH | combinación | **80 %** | 78 % | 74 % | **Sí, por poco** |

En validación (ene–abr 2026, época más seca): cloro 89 % (línea base 81 %), turbidez 71 % (62 %),
pH 88 % (84 %).

**Lo que muestra la comparación**

1. **Ninguno de los dos modelos es mejor en todo.**
   - La **LSTM** gana en turbidez: en validación llega a 71 %, contra 66 % de los árboles, y en
     prueba empata con la línea base mientras los árboles quedan 6 puntos por debajo. Al ver la
     secuencia completa de las últimas 48 h de lluvia y de la fuente, capta mejor cómo sube y baja
     la turbidez.
   - Los **árboles** ganan en cloro (89 % contra 87 % en validación).
   - La **combinación** es la mejor en pH, y la única que en prueba supera a la línea base.
2. **Entrenar solo con horas normales mejora a los árboles.** En validación suben de 88 % a 89 % en
   cloro y de 86 % a 87 % en pH respecto al notebook 06, y en prueba el pH pasa de 74 % a 78 %.
   Los cambios bruscos de los eventos son ruido para un pronóstico a 24 h, y de ellos se encarga el
   detector.
3. **La red neuronal se sobreajusta rápido.** En las curvas de aprendizaje el error de validación
   deja de bajar a las 2–4 épocas mientras el de entrenamiento sigue bajando. Hay pocos datos
   independientes (16 meses) para una red, aunque sea pequeña (17 687 parámetros). La parada
   temprana y el promedio de 3 entrenamientos lo controlan.
4. **La época lluviosa es el problema.** Entre validación y prueba la turbidez cae de 71 % a 42 % y
   el pH de 88 % a 80 %, igual que la línea base. Los dos modelos ven la lluvia que ya cayó, pero
   no la que viene. En el notebook 06, con lluvia futura perfecta, la turbidez llega al 90 %: el
   pronóstico de lluvia es la pieza que falta, y la LSTM es el modelo natural para recibirlo como
   una secuencia más (las próximas 24 h de lluvia pronosticada).
5. **Durante eventos los resultados no cambian mucho** (en prueba: cloro 50 %, pH 56 %, turbidez
   19 %), como se esperaba: es trabajo del detector.

**Sistema de pronóstico que queda (por ahora)**

- **Cloro:** árboles.
- **Turbidez:** LSTM, promedio de 3 entrenamientos.
- **pH:** promedio de árboles y LSTM.

Queda guardado en `modelos/pronostico_24h_final.joblib`, más los tres archivos `.keras` de la red.
El servicio en tiempo real debe cargar este, no el del notebook 06.

**Siguiente paso:** agregar a los dos modelos la lluvia pronosticada de las próximas 24 h
(pronósticos pasados de Open-Meteo para entrenar) y las ventanas de lluvia del notebook 04, y
probar si quitar la época del año (`anio_sin`, `anio_cos`) ayuda.
